# Stage 2: Cryptojacking Classification (Fine-Tuning)

Once you have gathered more cryptojacking binaries and disassembled them into `e:\Thesis\cryptojacking`, run this notebook.

This notebook will:
1. **Balance** the dataset (taking an equal number of benign and malware samples).
2. **Tokenize** using our newly trained Domain-Adapted GraphCodeBERT model.
3. **Add a Classification Head** (`num_labels=2`) on top of the pre-trained weights.
4. **Train** and save the final malware classifier.

In [1]:
# Ensure you have the required metrics library:
# %pip install evaluate scikit-learn

### 1. Data Loading and Balancing

In [2]:
import os
import glob
import json
import random
from datasets import Dataset

COMBINED_OUTPUT_DIR = r'e:\Thesis\combined_output'

def generate_data(data_dir, max_per_class=300):
    all_files = glob.glob(os.path.join(data_dir, '*.json'))
    random.shuffle(all_files)
    
    benign_count = 0
    malware_count = 0
    failed_files = 0
    
    for f in all_files:
        if benign_count >= max_per_class and malware_count >= max_per_class:
            break
        try:
            label = None
            asm = ""
            
            # Use the streaming parser to avoid loading gigabytes of graph data
            with open(f, 'r', encoding='utf-8') as file:
                for line in file:
                    stripped = line.strip()
                    
                    if stripped.startswith('"label":'):
                        label = int(stripped.split(':')[1].strip().strip(','))
                        
                    elif stripped.startswith('"asm":'):
                        if stripped.endswith(','):
                            stripped = stripped[:-1]
                        
                        mini_json = "{" + stripped + "}"
                        # Only keep the first 50,000 chars and immediately discard the rest
                        asm = json.loads(mini_json)["asm"][:50000]
                        # Break immediately to avoid reading the gigabytes of graph arrays!
                        break
            
            if not asm or label is None:
                continue
            
            if label == 0 and benign_count < max_per_class:
                benign_count += 1
                yield {'text': asm, 'label': 0}
            elif label == 1 and malware_count < max_per_class:
                malware_count += 1
                yield {'text': asm, 'label': 1}
                
        except Exception:
            failed_files += 1
    
    if failed_files > 0:
        print(f"Warning: {failed_files} files failed to parse and were skipped.")
    
    print(f"Loaded {benign_count} benign and {malware_count} cryptojacking samples.")

dataset = Dataset.from_generator(generate_data, gen_kwargs={"data_dir": COMBINED_OUTPUT_DIR, "max_per_class": 300})

# Split before tokenizing to prevent data leakage
data_list = [dataset[i] for i in range(len(dataset))]
random.shuffle(data_list)
split_idx = int(len(data_list) * 0.8)
train_data = Dataset.from_list(data_list[:split_idx])
eval_data = Dataset.from_list(data_list[split_idx:])

print(f"Train size: {len(train_data)} | Eval size: {len(eval_data)}")

C:\Users\cjrea\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 586 examples [00:34, 16.88 examples/s]


Loaded 300 benign and 286 cryptojacking samples.
Train size: 468 | Eval size: 118


### 2. Tokenization
We load the tokenizer from our Stage 1 model. Chunking to 512 tokens.

In [6]:
from transformers import AutoTokenizer
import os

# Point this to your Stage 1 final folder
MODEL_PATH = "./gcb-assembly-final"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

def tokenize_function(examples):
    # Tokenize with overflow to capture the ENTIRE binary assembly in multiple 512 chunks
    tokenized = tokenizer(
        examples["text"], # We already truncated to 150k max in the loading step!
        truncation=True,
        padding="max_length", # CRITICAL FIX: Pad the final chunks so they are exactly 512 tokens
        max_length=512,
        return_overflowing_tokens=True,
        stride=16
    )
    
    # Map the original class label to each newly generated overlapping chunk
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    labels = []
    for mapping_index in sample_mapping:
        labels.append(examples["label"][mapping_index])
    
    tokenized["label"] = labels
    return tokenized

# We drop the old text and label columns, letting the new tokenized ones take over
# CRITICAL OOM FIX: 
# 1. Removed `num_proc`: Multiprocessing on Windows duplicates memory per-core cleanly causing massive RAM spikes.
# 2. Reduced `batch_size=50`: Usually 1000, reducing to 50 prevents the batch loader from creating enormous token arrays in memory during mapping.
train_dataset = train_data.map(
    tokenize_function, 
    batched=True, 
    batch_size=50, 
    remove_columns=["text", "label"]
)
eval_dataset = eval_data.map(
    tokenize_function, 
    batched=True, 
    batch_size=50, 
    remove_columns=["text", "label"]
)

Map: 100%|██████████| 118/118 [00:02<00:00, 58.51 examples/s]


### 3. Model Initialization and Metrics
Notice we are using `AutoModelForSequenceClassification` now, not `AutoModelForMaskedLM`.

In [7]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import torch
import evaluate
import numpy as np

# Load evaluation metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    # Calculate Accuracy 
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    
    # Calculate Precision, Recall, and F1 prioritizing Binary classification for Malware
    # This ensures we don't just default to generic averages, letting us see missed malware
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="binary")["f1"]
    precision = precision_metric.compute(predictions=predictions, references=labels, average="binary")["precision"]
    recall = recall_metric.compute(predictions=predictions, references=labels, average="binary")["recall"]
    
    return {
        "accuracy": acc, 
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# Load Pre-Trained Weights, but with a Classification Head for 2 labels
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, 
    num_labels=2
)

# Note: It is expected to see a warning here about "some weights not being initialized" 
# because the LM Head from Stage 1 is being thrown away and replaced by a randomly initialized Classification Head.

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at ./gcb-assembly-final and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 4. Training (Fine-tuning the Classifier)

In [8]:
training_args = TrainingArguments(
    output_dir="./gcb-stage2-classification",
    overwrite_output_dir=False,       # Changed from True to prevent nuking existing checkpoints
    num_train_epochs=10,              # Increased max epochs slightly, relying on EarlyStopping
    per_device_train_batch_size=8,    
    per_device_eval_batch_size=8,
    warmup_ratio=0.06,                # Changed from warmup_steps for consistency with Stage 1
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,               # Keep only the last 2 checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="f1",       # Use F1 score for early stopping/best model instead of loss
    fp16=True,                        
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Stop if F1 plateaus for 3 epochs
)

print("Starting Training...")
trainer.train()

Starting Training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.091700,0.314906,0.927054,0.931575,0.950645,0.913254
2,0.113800,0.562115,0.912260,0.914294,0.974977,0.860722
3,0.078300,0.377745,0.934671,0.938381,0.963131,0.914871
4,0.000100,0.498701,0.938773,0.943360,0.949019,0.937769
5,0.000000,0.427876,0.945657,0.950540,0.940881,0.960399
6,0.000100,0.442411,0.943020,0.947820,0.943895,0.951778
7,0.000000,0.497896,0.944632,0.948711,0.955714,0.941810
8,0.000000,0.483514,0.949758,0.953879,0.952215,0.955550
9,0.000000,0.563327,0.944485,0.948442,0.957955,0.939116
10,0.000000,0.618060,0.943167,0.947038,0.959878,0.934537


TrainOutput(global_step=33760, training_loss=0.04786723364250496, metrics={'train_runtime': 7951.184, 'train_samples_per_second': 33.967, 'train_steps_per_second': 4.246, 'total_flos': 7.10610338316288e+16, 'train_loss': 0.04786723364250496, 'epoch': 10.0})

### 5. Final Save

In [9]:
# Save the final classifier to disk
FINAL_STAGE2_DIR = "./gcb-stage2-final"
trainer.save_model(FINAL_STAGE2_DIR)
tokenizer.save_pretrained(FINAL_STAGE2_DIR)

print(f"Success! Your Malware Classifier is saved at {FINAL_STAGE2_DIR}")

Success! Your Malware Classifier is saved at ./gcb-stage2-final


### 6. Inference (Testing on Unseen Data)

We will now load your newly saved classifier from disk and test it against completely new, unseen malware samples in your `testing_output` folder.

In [10]:
import os
import glob
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

TEST_DIR = r"e:\Thesis\testing_output"
SAVED_MODEL_DIR = "./gcb-stage2-final"

print("[*] Loading final malware classifier from disk...")
tokenizer = AutoTokenizer.from_pretrained(SAVED_MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(SAVED_MODEL_DIR)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

# Scan the testing folder for disassembled JSONs
test_files = glob.glob(os.path.join(TEST_DIR, "*.json"))
print(f"[*] Found {len(test_files)} testing files.\n")

for f in test_files:
    try:
        asm = ""
        actual_label = "Unknown"
        
        # Memory-efficiently stream grab the true label and first 50,000 chars of ASM
        with open(f, 'r', encoding='utf-8') as file:
            for line in file:
                stripped = line.strip()
                if stripped.startswith('"label_name":'):
                    # e.g., "label_name": "cryptojacking",
                    actual_label = stripped.split(':')[1].strip().strip('",')
                elif stripped.startswith('"asm":'):
                    if stripped.endswith(','):
                        stripped = stripped[:-1]
                    mini_json = "{" + stripped + "}"
                    asm = json.loads(mini_json)["asm"][:50000]
                    break
        
        if not asm:
            print(f"[SKIP] {os.path.basename(f)[:20]}... - No assembly found.")
            continue
            
        # Tokenize the assembly snippet for inference
        # We slice up to 512 tokens to check the behavior of the entrypoint and unpacking stubs
        inputs = tokenizer(asm, return_tensors="pt", truncation=True, max_length=512).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            # The model outputs raw logits. Softmax/Argmax finds the highest probability class
            prediction_id = torch.argmax(outputs.logits, dim=-1).item()
            
        label_map = {0: "benign", 1: "cryptojacking"}
        pred_name = label_map[prediction_id]
        
        # Check if the model got it right (assuming the true label exists in the testing JSON)
        match = "✅" if actual_label.lower() == pred_name.lower() else "❌"
        if actual_label == "Unknown": 
            match = "❓"
            
        filename_short = os.path.basename(f)
        if len(filename_short) > 25: filename_short = filename_short[:22] + "..."
            
        print(f"File: {filename_short:<25} | True: {actual_label:<15} | Pred: {pred_name:<15} | {match}")
        
    except Exception as e:
        print(f"[ERROR] failed on {os.path.basename(f)}: {e}")

[*] Loading final malware classifier from disk...
[*] Found 1 testing files.

File: 411b6a3a552290239c1af8... | True: cryptojacking   | Pred: cryptojacking   | ✅
